# Transformer 从零搭建

本 Notebook 从最基础的组件开始，逐步搭建完整的 Transformer 模型。

## 目录
1. [多头自注意力机制](#1-多头自注意力机制)
2. [位置编码](#2-位置编码)
3. [前馈网络与残差连接](#3-前馈网络与残差连接)
4. [Encoder Layer](#4-encoder-layer)
5. [Decoder Layer](#5-decoder-layer)
6. [完整 Transformer](#6-完整-transformer)
7. [实战：字符级文本生成](#7-实战字符级文本生成)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

---
## 1. 多头自注意力机制

核心公式：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

多头注意力将 Q/K/V 投影到 $h$ 个子空间，分别计算注意力后拼接。

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """缩放点积注意力"""
    def __init__(self, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(self, Q, K, V, mask=None):
        d_k = Q.size(-1)
        # Q: [B, H, L_q, d_k], K: [B, H, L_k, d_k]
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        output = torch.matmul(attn_weights, V)
        return output, attn_weights


class MultiHeadAttention(nn.Module):
    """多头自注意力"""
    def __init__(self, d_model=512, n_heads=8, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model 必须能被 n_heads 整除"

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        # Q/K/V 线性投影 + 输出投影
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention(dropout)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)

        # 线性投影并拆分为多头: [B, L, d_model] -> [B, H, L, d_k]
        Q = self.W_q(Q).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(K).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(V).view(B, -1, self.n_heads, self.d_k).transpose(1, 2)

        out, attn_weights = self.attention(Q, K, V, mask)

        # 拼接多头: [B, H, L, d_k] -> [B, L, d_model]
        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        out = self.W_o(out)
        return out, attn_weights


# ---- 验证 ----
mha = MultiHeadAttention(d_model=64, n_heads=4)
x = torch.randn(2, 10, 64)  # [batch, seq_len, d_model]
out, attn = mha(x, x, x)
print(f'Input:  {x.shape}')
print(f'Output: {out.shape}')
print(f'Attention weights: {attn.shape}')  # [B, H, L, L]

In [ ]:
# 可视化注意力权重
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i in range(4):
    axes[i].imshow(attn[0, i].detach().numpy(), cmap='Blues')
    axes[i].set_title(f'Head {i+1}')
    axes[i].set_xlabel('Key')
    axes[i].set_ylabel('Query')
plt.suptitle('Multi-Head Attention Weights')
plt.tight_layout()
plt.show()

---
## 2. 位置编码

Transformer 没有递归/卷积结构，需要显式注入位置信息。

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    """正弦位置编码"""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [B, L, d_model]
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class LearnedPositionalEncoding(nn.Module):
    """可学习位置编码"""
    def __init__(self, d_model, max_len=512):
        super().__init__()
        self.pe = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


# 可视化正弦位置编码
pe = PositionalEncoding(d_model=128, max_len=200, dropout=0)
pe_matrix = pe.pe[0, :, :64].detach().numpy()  # 取前64维

plt.figure(figsize=(12, 4))
plt.imshow(pe_matrix.T, aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Sinusoidal Positional Encoding')
plt.show()

---
## 3. 前馈网络与残差连接

每个子层使用残差连接 + LayerNorm：
$$\text{Output} = \text{LayerNorm}(x + \text{Sublayer}(x))$$

FFN 是两层线性变换 + 激活：
$$\text{FFN}(x) = \text{ReLU}(xW_1 + b_1)W_2 + b_2$$

In [ ]:
class PositionwiseFeedForward(nn.Module):
    """位置级前馈网络"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class SublayerConnection(nn.Module):
    """残差连接 + LayerNorm (Pre-LN)"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))


# 对比 Pre-LN vs Post-LN
print("""Pre-LN (GPT风格, 训练更稳定):
  output = x + Sublayer(LayerNorm(x))

Post-LN (原版Transformer):
  output = LayerNorm(x + Sublayer(x))

现代实践几乎统一使用 Pre-LN，因为它避免了训练初期的梯度爆炸问题。""")

---
## 4. Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    """单个 Encoder 层"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)

    def forward(self, x, mask=None):
        # 自注意力 + 残差
        x = self.sublayer1(x, lambda x: self.self_attn(x, x, x, mask)[0])
        # FFN + 残差
        x = self.sublayer2(x, self.ffn)
        return x


# 验证
enc_layer = EncoderLayer(d_model=64, n_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)
out = enc_layer(x)
print(f'EncoderLayer: {x.shape} -> {out.shape}')
print(f'参数量: {sum(p.numel() for p in enc_layer.parameters()):,}')

---
## 5. Decoder Layer

Decoder 多了一个交叉注意力层（Cross-Attention），并使用因果掩码（Causal Mask）防止看到未来信息。

In [ ]:
class DecoderLayer(nn.Module):
    """单个 Decoder 层"""
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = PositionwiseFeedForward(d_model, d_ff, dropout)
        self.sublayer1 = SublayerConnection(d_model, dropout)
        self.sublayer2 = SublayerConnection(d_model, dropout)
        self.sublayer3 = SublayerConnection(d_model, dropout)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        # 带因果掩码的自注意力
        x = self.sublayer1(x, lambda x: self.self_attn(x, x, x, tgt_mask)[0])
        # 交叉注意力: Q 来自 decoder, K/V 来自 encoder
        x = self.sublayer2(x, lambda x: self.cross_attn(x, enc_output, enc_output, src_mask)[0])
        # FFN
        x = self.sublayer3(x, self.ffn)
        return x


def generate_causal_mask(seq_len):
    """生成因果掩码（下三角矩阵）"""
    mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0).unsqueeze(0)
    return mask  # [1, 1, L, L]


# 可视化因果掩码
mask = generate_causal_mask(8)
plt.figure(figsize=(4, 3))
plt.imshow(mask[0, 0].numpy(), cmap='gray')
plt.title('Causal Mask')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.show()
print('1 = 可见, 0 = 遮蔽（不允许看到未来）')

# 验证
dec_layer = DecoderLayer(d_model=64, n_heads=4, d_ff=256)
x = torch.randn(2, 8, 64)
enc_out = torch.randn(2, 12, 64)
tgt_mask = generate_causal_mask(8)
out = dec_layer(x, enc_out, tgt_mask=tgt_mask)
print(f'\nDecoderLayer: {x.shape} + enc{enc_out.shape} -> {out.shape}')

---
## 6. 完整 Transformer

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)


class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len=512, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.projection = nn.Linear(d_model, vocab_size)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, enc_output, src_mask, tgt_mask)
        x = self.norm(x)
        return self.projection(x)


class Transformer(nn.Module):
    """完整 Seq2Seq Transformer"""
    def __init__(self, src_vocab, tgt_vocab, d_model=256, n_heads=8, d_ff=1024,
                 n_enc_layers=3, n_dec_layers=3, max_len=512, dropout=0.1):
        super().__init__()
        self.encoder = TransformerEncoder(src_vocab, d_model, n_heads, d_ff, n_enc_layers, max_len, dropout)
        self.decoder = TransformerDecoder(tgt_vocab, d_model, n_heads, d_ff, n_dec_layers, max_len, dropout)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_output = self.encoder(src, src_mask)
        if tgt_mask is None:
            tgt_mask = generate_causal_mask(tgt.size(1)).to(tgt.device)
        output = self.decoder(tgt, enc_output, src_mask, tgt_mask)
        return output


# 构建一个小型 Transformer
model = Transformer(src_vocab=1000, tgt_vocab=1000, d_model=128, n_heads=4, d_ff=512, n_enc_layers=2, n_dec_layers=2)
src = torch.randint(0, 1000, (4, 20))  # [B, src_len]
tgt = torch.randint(0, 1000, (4, 15))  # [B, tgt_len]
out = model(src, tgt)
print(f'Source: {src.shape}')
print(f'Target: {tgt.shape}')
print(f'Output: {out.shape}')  # [B, tgt_len, tgt_vocab]
total_params = sum(p.numel() for p in model.parameters())
print(f'总参数量: {total_params:,}')

---
## 7. 实战：字符级文本生成

用一个小型 Transformer Decoder（GPT 风格）学习生成文本。

In [ ]:
# 准备一个小数据集
text = """The transformer architecture has revolutionized natural language processing. 
Self-attention allows the model to weigh the importance of different words in a sequence. 
The key innovation is the ability to process all positions in parallel, unlike recurrent networks. 
Multi-head attention enables the model to attend to information from different representation subspaces. 
Position encoding provides the model with sequence order information. 
The encoder-decoder structure is used for sequence-to-sequence tasks like translation. 
Decoder-only models like GPT are widely used for text generation. 
Encoder-only models like BERT excel at understanding tasks such as classification."""

# 构建字符级词表
chars = sorted(set(text))
vocab_size = len(chars)
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

print(f'词表大小: {vocab_size}')
print(f'字符: {".".join(chars)}')

# 编码整个文本
data = torch.tensor([char_to_idx[c] for c in text], dtype=torch.long)
print(f'数据长度: {len(data)} 字符')

In [ ]:
class GPTLikeModel(nn.Module):
    """简易 GPT 模型 (Decoder-only Transformer)"""
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=512, n_layers=2, max_len=256, dropout=0.1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, L = x.shape
        mask = generate_causal_mask(L).to(x.device)
        x = self.embedding(x)
        x = self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x, mask)
        x = self.norm(x)
        return self.head(x)


def get_batch(data, batch_size, block_size):
    """随机采样一个 batch"""
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y


def generate(model, idx, max_new_tokens, temperature=0.8, top_k=None):
    """自回归生成"""
    model.eval()
    for _ in range(max_new_tokens):
        # 取最后 block_size 个 token
        idx_cond = idx[:, -256:]
        logits = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k is not None:
            v, _ = torch.topk(logits, top_k)
            logits[logits < v[:, [-1]]] = float('-inf')
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, idx_next], dim=1)
    return idx

In [ ]:
# 训练
model = GPTLikeModel(vocab_size, d_model=64, n_heads=4, d_ff=256, n_layers=2).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

block_size = 64
batch_size = 16
losses = []

data = data.to(device)
model.train()
for step in range(500):
    x, y = get_batch(data, batch_size, block_size)
    logits = model(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    losses.append(loss.item())
    if step % 100 == 0:
        print(f'Step {step:4d} | Loss: {loss.item():.4f}')

# 绘制 Loss 曲线
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.show()

In [ ]:
# 生成示例
context = torch.tensor([[char_to_idx['T']]], device=device)
generated = generate(model, context, max_new_tokens=200, temperature=0.7, top_k=10)
text_gen = ''.join([idx_to_char[i] for i in generated[0].tolist()])
print('=== 生成结果 ===')
print(text_gen)

---
## 关键知识点总结

| 组件 | 作用 | 复杂度 |
|------|------|--------|
| 自注意力 | 捕捉序列内任意位置的依赖关系 | $O(n^2 \cdot d)$ |
| 多头注意力 | 从多个子空间学习不同的注意力模式 | $O(n^2 \cdot d)$ |
| 位置编码 | 为无序的注意力注入位置信息 | $O(n \cdot d)$ |
| FFN | 非线性特征变换 | $O(n \cdot d \cdot d_{ff})$ |
| 因果掩码 | 防止 Decoder 看到未来信息 | - |
| 残差+LayerNorm | 稳定深层网络的训练 | $O(n \cdot d)$ |

**Transformer 的核心瓶颈：自注意力的 $O(n^2)$ 复杂度，这也催生了后续诸多线性注意力、SSM 等替代方案。**